# Step 4 — MPS SLAM Validation

Reproduces **Section IV-B** of the paper.

Confirms: quality=0.996, position stability=0.4mm, yaw stability=0.16°, all 117 recordings valid.

## Configuration

**Change `BASE` below to match your machine before running anything else.**
All other paths are derived automatically.

In [ ]:
from pathlib import Path

# ── Set your base path here ───────────────────────────────────────────────
# Change this to where your esas_project folder is on your machine.
# Everything else is derived automatically.
BASE = Path('/path/to/your/esas_project')  # <-- set this
# ─────────────────────────────────────────────────────────────────────────

RECORDINGS = BASE / 'recordings'
ESC50_DIR  = BASE / 'ESC-50'
PANNS_CKPT = BASE / 'panns_data' / 'Cnn14_mAP=0.431.pth'
MODELS_DIR = Path('models')  # saved inside esas_clean
RESULTS_DIR = Path('results')  # saved inside esas_clean
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
print(f'BASE:       {BASE}')
print(f'Recordings: {RECORDINGS.exists()}')
print(f'ESC-50:     {ESC50_DIR.exists()}')
print(f'PANNs:      {PANNS_CKPT.exists()}')

In [ ]:
import math, re
import numpy as np
import pandas as pd

def quat_to_yaw(qx, qy, qz, qw):
    siny = 2*(qw*qz + qx*qy)
    cosy = 1 - 2*(qy*qy + qz*qz)
    return math.degrees(math.atan2(siny, cosy))

def parse_angle(name):
    m = re.search(r'(-?\d+)_\d+$', name.replace('mps_','').replace('_vrs',''))
    return int(m.group(1)) if m else None

mps_dirs = sorted([d for d in RECORDINGS.iterdir()
                   if d.is_dir() and d.name.startswith('mps_')])
print(f'Found {len(mps_dirs)} MPS folders')

In [ ]:
rows = []
for d in mps_dirs:
    traj = d / 'slam' / 'open_loop_trajectory.csv'
    if not traj.exists(): continue
    angle = parse_angle(d.name)
    if angle is None: continue
    try:
        df  = pd.read_csv(traj, comment='#')
        q   = float(df['quality_score'].mean())
        ps  = float(np.sqrt(df['tx_odometry_device'].std()**2 +
                            df['ty_odometry_device'].std()**2 +
                            df['tz_odometry_device'].std()**2))
        yaws = [quat_to_yaw(r.qx_odometry_device,r.qy_odometry_device,
                            r.qz_odometry_device,r.qw_odometry_device)
                for _,r in df.iterrows()]
        rows.append({'recording':d.name,'angle':angle,
                     'quality':round(q,3),'pos_std_mm':round(ps*1000,2),
                     'yaw_std':round(float(np.std(yaws)),3),'valid':q>=0.4 and ps<0.05})
    except Exception as e:
        print(f'Error {d.name}: {e}')

df = pd.DataFrame(rows)
print(f'Analysed: {len(df)} recordings')

In [ ]:
print('='*50)
print('  MPS SLAM Validation — Section IV-B')
print('='*50)
print(f'  Total:          {len(df)}')
print(f'  Valid:          {df["valid"].sum()}')
print(f'  Mean quality:   {df["quality"].mean():.3f}  (paper: 0.996)')
print(f'  Pos stability:  {df["pos_std_mm"].mean():.1f} mm  (paper: 0.4 mm)')
print(f'  Yaw stability:  {df["yaw_std"].mean():.2f}\u00b0  (paper: 0.16\u00b0)')
print('='*50)
df.to_csv(RESULTS_DIR / 'mps_results.csv', index=False)
print(f'Saved: {RESULTS_DIR}/mps_results.csv')